In [6]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as Estimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    precision_score, recall_score, f1_score, roc_auc_score
)

In [ ]:
N_SAMPLES    = 2000
N_PCA        = 4
FIRE_RATIO   = 0.25   # fires will be ~25% of dataset (adjust to match real distribution)

df_full = pd.read_csv("features.csv")

fire   = df_full[df_full["target"] == 1]
nofire = df_full[df_full["target"] == 0]

# Keep all fires (up to budget), fill the rest with no-fires
n_fire   = min(len(fire),   int(N_SAMPLES * FIRE_RATIO))
n_nofire = min(len(nofire), N_SAMPLES - n_fire)

df = pd.concat([
    fire.sample(n=n_fire,   random_state=42),
    nofire.sample(n=n_nofire, random_state=42),
]).sample(frac=1, random_state=42).reset_index(drop=True)

feature_cols = ["month_sin","month_cos","year","avg_tmax_c","temp_range",
                "tot_prcp_mm","dryness_3m_avg","latitude","longitude"]
X = df[feature_cols].values.astype(float)
y = df["target"].values.astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

std_scaler = StandardScaler()
X_train_s  = std_scaler.fit_transform(X_train)
X_test_s   = std_scaler.transform(X_test)

pca = PCA(n_components=N_PCA, random_state=42)
X_train_p = pca.fit_transform(X_train_s)
X_test_p  = pca.transform(X_test_s)

angle_scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
X_train_a = angle_scaler.fit_transform(X_train_p)
X_test_a  = angle_scaler.transform(X_test_p)

print(f"Train: {X_train_a.shape}  Test: {X_test_a.shape}")
print(f"Fire rate — train: {y_train.mean():.1%}  test: {y_test.mean():.1%}")
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}")

Train: (1700, 4)  Test: (300, 4)
Fire rate — train: 0.0%  test: 0.0%
PCA explained variance: 80.9%


In [8]:
N_QUBITS = N_PCA   # 4 qubits — simulation is O(2^n), so 4 vs 9 is ~32x faster
REPS     = 2       # 2 reps with data re-uploading between them

x_params = ParameterVector("x", N_QUBITS)
n_weights = N_QUBITS * 2 * REPS          # RY + RZ per qubit per rep = 16 params
theta    = ParameterVector("θ", n_weights)

qc = QuantumCircuit(N_QUBITS)

# Initial superposition
for i in range(N_QUBITS):
    qc.h(i)

t = 0
for r in range(REPS):
    # Data re-uploading: encode x at the start of every rep
    # (same symbolic params reused — Qiskit handles this correctly)
    for i in range(N_QUBITS):
        qc.ry(x_params[i], i)

    # Trainable rotations
    for q in range(N_QUBITS):
        qc.ry(theta[t], q); t += 1
        qc.rz(theta[t], q); t += 1

    # Ring entanglement — creates correlations between all qubits
    for q in range(N_QUBITS - 1):
        qc.cx(q, q + 1)
    qc.cx(N_QUBITS - 1, 0)

print(f"Qubits: {N_QUBITS}  |  Trainable θ params: {n_weights}  |  Reps: {REPS}")
qc.decompose().draw("text")

Qubits: 4  |  Trainable θ params: 16  |  Reps: 2


global phase: (-0.5)*θ[1] - 0.5*θ[3] - 0.5*θ[5] - 0.5*θ[7] - 0.5*θ[9] - 0.5*θ[11] - 0.5*θ[13] - 0.5*θ[15]
     ┌────────────┐┌─────────────┐┌─────────────┐┌─────────┐          »
q_0: ┤ U(π/2,0,π) ├┤ R(x[0],π/2) ├┤ R(θ[0],π/2) ├┤ P(θ[1]) ├──■───────»
     ├────────────┤├─────────────┤├─────────────┤├─────────┤┌─┴─┐     »
q_1: ┤ U(π/2,0,π) ├┤ R(x[1],π/2) ├┤ R(θ[2],π/2) ├┤ P(θ[3]) ├┤ X ├──■──»
     ├────────────┤├─────────────┤├─────────────┤├─────────┤└───┘┌─┴─┐»
q_2: ┤ U(π/2,0,π) ├┤ R(x[2],π/2) ├┤ R(θ[4],π/2) ├┤ P(θ[5]) ├─────┤ X ├»
     ├────────────┤├─────────────┤├─────────────┤├─────────┤     └───┘»
q_3: ┤ U(π/2,0,π) ├┤ R(x[3],π/2) ├┤ R(θ[6],π/2) ├┤ P(θ[7]) ├──────────»
     └────────────┘└─────────────┘└─────────────┘└─────────┘          »
«                                    ┌───┐┌─────────────┐ ┌─────────────┐ »
«q_0: ───────────────────────────────┤ X ├┤ R(x[0],π/2) ├─┤ R(θ[8],π/2) ├─»
«     ┌─────────────┐┌──────────────┐└─┬─┘└─┬──────────┬┘ └─────────────┘ »
«q_1: ┤ R(x[1],π/2) ├┤ R(θ[10],π/2) ├──┼────┤ P(θ[11]) ├──────────────────»
«     └─────────────┘├─────────────┬┘  │  ┌─┴──────────┴─┐  ┌──────────┐  »
«q_2: ───────■───────┤ R(x[2],π/2) ├───┼──┤ R(θ[12],π/2) ├──┤ P(θ[13]) ├──»
«          ┌─┴─┐     └─────────────┘   │  ├─────────────┬┘┌─┴──────────┴─┐»
«q_3: ─────┤ X ├───────────────────────■──┤ R(x[3],π/2) ├─┤ R(θ[14],π/2) ├»
«          └───┘                          └─────────────┘ └──────────────┘»
«     ┌─────────┐                ┌───┐
«q_0: ┤ P(θ[9]) ├───■────────────┤ X ├
«     └─────────┘ ┌─┴─┐          └─┬─┘
«q_1: ────────────┤ X ├──■─────────┼──
«                 └───┘┌─┴─┐       │  
«q_2: ─────────────────┤ X ├──■────┼──
«     ┌──────────┐     └───┘┌─┴─┐  │  
«q_3: ┤ P(θ[15]) ├──────────┤ X ├──■──
«     └──────────┘          └───┘

In [10]:
# 7 observables as a LIST — EstimatorQNN returns one expectation value per entry → shape (batch, 7)
# A single SparsePauliOp with multiple terms sums them into one scalar; a list gives separate outputs.
# Qiskit Pauli strings are right-to-left: rightmost char = qubit 0
observables = [
    SparsePauliOp.from_list([("IIIZ", 1.0)]),   # Z on qubit 0
    SparsePauliOp.from_list([("IIZI", 1.0)]),   # Z on qubit 1
    SparsePauliOp.from_list([("IZII", 1.0)]),   # Z on qubit 2
    SparsePauliOp.from_list([("ZIII", 1.0)]),   # Z on qubit 3
    SparsePauliOp.from_list([("IIZZ", 1.0)]),   # ZZ on qubits 0,1 — entanglement signal
    SparsePauliOp.from_list([("IZZI", 1.0)]),   # ZZ on qubits 1,2
    SparsePauliOp.from_list([("ZZII", 1.0)]),   # ZZ on qubits 2,3
]
N_OBS = len(observables)  # 7

estimator = Estimator()
qnn = EstimatorQNN(
    circuit=qc,
    estimator=estimator,
    observables=observables,
    input_params=list(x_params),
    weight_params=list(theta),
)

# Hybrid model: QNN (quantum feature extractor) + classical linear head
class HybridModel(nn.Module):
    def __init__(self, qnn, n_obs):
        super().__init__()
        self.qnn  = TorchConnector(qnn)
        self.head = nn.Linear(n_obs, 1)

    def forward(self, x):
        x = self.qnn(x)      # (batch, 7)
        return self.head(x)  # (batch, 1) — raw logit

torch.manual_seed(42)
model = HybridModel(qnn, N_OBS)
n_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable params: {n_params}  (QNN θ: {n_weights}, head: {N_OBS+1})")

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


Total trainable params: 24  (QNN θ: 16, head: 8)


In [11]:
EPOCHS     = 50
BATCH_SIZE = 32   # mini-batching: QNN processes 32 samples per forward pass (batched by Estimator)
LR         = 0.02

X_tr = torch.tensor(X_train_a, dtype=torch.float32)
y_tr = torch.tensor(y_train,   dtype=torch.float32).view(-1, 1)
X_te = torch.tensor(X_test_a,  dtype=torch.float32)
y_te = torch.tensor(y_test,    dtype=torch.float32).view(-1, 1)

loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)

loss_fn   = nn.BCEWithLogitsLoss()  # sigmoid + BCE — correct for binary classification
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
    scheduler.step()

    if epoch % 5 == 0 or epoch == 1:
        avg = total_loss / len(X_tr)
        lr_now = scheduler.get_last_lr()[0]
        print(f"epoch={epoch:3d}  loss={avg:.4f}  lr={lr_now:.4f}")

epoch=  1  loss=0.5648  lr=0.0200
epoch=  5  loss=0.0606  lr=0.0195
epoch= 10  loss=0.0216  lr=0.0181


KeyboardInterrupt: 

In [ ]:
model.eval()
with torch.no_grad():
    logits   = model(X_te).numpy().ravel()

y_prob    = 1 / (1 + np.exp(-logits))   # sigmoid → predicted probabilities
y_pred    = (y_prob >= 0.5).astype(int)
y_true    = y_test.astype(int)

# Regression metrics on predicted probability vs binary label (Brier-style)
mae  = mean_absolute_error(y_true, y_prob)
rmse = np.sqrt(mean_squared_error(y_true, y_prob))
r2   = r2_score(y_true, y_prob)

# Classification metrics
precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)

n_classes = len(np.unique(y_true))
auc_roc   = roc_auc_score(y_true, y_prob) if n_classes == 2 else float("nan")

print(f"{'Metric':<12} {'Value':>8}")
print("-" * 22)
print(f"{'MAE':<12} {mae:>8.4f}")
print(f"{'RMSE':<12} {rmse:>8.4f}")
print(f"{'R2':<12} {r2:>8.4f}")
print(f"{'Precision':<12} {precision:>8.4f}")
print(f"{'Recall':<12} {recall:>8.4f}")
print(f"{'F1':<12} {f1:>8.4f}")
print(f"{'AUC_ROC':<12} {auc_roc:>8.4f}")